# 02. Statystyki opisowe BNB/USDT

Niezależne obliczenia kontrolne na dziennych zamknięciach Binance od 1.01.2023 UTC. Do statystyk dziennych wchodzą tylko zwroty między kolejnymi dniami wybranego okresu. Zwroty i zmienność podano jako ułamki (np. 0,05 = 5%), o ile nie zaznaczono inaczej.

In [1]:
import numpy as np
import pandas as pd
from market_data import daily_price_ratios, download

def statistics(prices):
    prices = prices.sort_index().astype(float)
    ratios = daily_price_ratios(prices).dropna()
    simple = ratios - 1
    log = np.log(ratios)
    result = {"liczba cen": len(prices), "liczba zwrotów dziennych": len(simple)}
    if len(prices) < 2:
        return result
    result["zwrot za okres"] = prices.iloc[-1] / prices.iloc[0] - 1
    days = (prices.index[-1] - prices.index[0]).days
    result["CAGR"] = (prices.iloc[-1] / prices.iloc[0]) ** (365 / days) - 1 if days >= 365 else np.nan
    running_peak = prices.cummax()
    drawdown = prices / running_peak - 1
    trough = drawdown.idxmin()
    if drawdown.loc[trough] < 0:
        peak_val = prices.loc[:trough].max()
        peak = prices.loc[:trough].loc[lambda x: x == peak_val].index[-1]
        recovery = prices.loc[trough:].iloc[1:].loc[lambda x: x >= peak_val]
        result.update({"maksymalne obsunięcie": drawdown.loc[trough], "data szczytu UTC": peak.date(),
                       "data dołka UTC": trough.date(), "liczba dni od szczytu do dołka": (trough - peak).days,
                       "powrót do szczytu UTC": recovery.index[0].date() if len(recovery) else None})
    else:
        result.update({"maksymalne obsunięcie": 0.0, "data szczytu UTC": None,
                       "data dołka UTC": None, "liczba dni od szczytu do dołka": None, "powrót do szczytu UTC": None})
    if len(simple) >= 2:
        nonzero = simple[simple != 0]
        result.update({"średni zwrot prosty": simple.mean(), "mediana zwrotów prostych": simple.median(),
                       "najlepszy dzień": simple.max(), "data najlepszego dnia UTC": simple.idxmax().date(),
                       "najgorszy dzień": simple.min(), "data najgorszego dnia UTC": simple.idxmin().date(),
                       "odsetek dni wzrostowych": (nonzero > 0).mean() if len(nonzero) else np.nan,
                       "odchylenie logarytmicznych zwrotów (n-1)": log.std(ddof=1),
                       "zmienność roczna": log.std(ddof=1) * np.sqrt(365),
                       "skośność": log.skew() if len(log) >= 3 else np.nan,
                       "kurtoza": log.kurt() if len(log) >= 4 else np.nan})
    return result

## Kontrola na danych z rozdziału 11 metod

Sprawdzamy wyniki z dokładnością odpowiednią do wartości podanych w dokumencie. Dodatkowo sprawdzamy korelację, betę, skośność, kurtozę i zachowanie przy luce.

In [2]:
dates = pd.date_range("2023-01-01", periods=5, freq="D", tz="UTC")
prices = pd.Series([100, 110, 99, 108.9, 102.366], index=dates)
test = statistics(prices)
np.testing.assert_allclose([test["średni zwrot prosty"], test["mediana zwrotów prostych"],
                            test["odchylenie logarytmicznych zwrotów (n-1)"], test["zmienność roczna"],
                            test["zwrot za okres"], test["maksymalne obsunięcie"],
                            test["skośność"], test["kurtoza"]],
                           [0.01, 0.02, 0.1048185044, 2.0025547144, 0.02366, -0.10, -0.1468980258, -5.1579652449], atol=1e-9)
assert str(test["data szczytu UTC"]) == "2023-01-02" and str(test["data dołka UTC"]) == "2023-01-03"
assert test["liczba dni od szczytu do dołka"] == 1
assert test["powrót do szczytu UTC"] is None
logs = np.log(prices / prices.shift()).dropna()
np.testing.assert_allclose([logs.sum(), prices.rolling(3).mean().iloc[-1]],
                           [0.0233844402, 103.422], atol=1e-9)
bnb = np.array([.10, -.10, .10, -.06])
btc = np.array([.02, -.01, .03, -.02])
np.testing.assert_allclose([np.corrcoef(bnb, btc)[0, 1], np.cov(bnb, btc, ddof=1)[0, 1] / np.var(btc, ddof=1)],
                           [0.9317610191, 4.1176470588], atol=1e-9)
gap = statistics(prices.drop(dates[2]))
assert gap["liczba zwrotów dziennych"] == 2
zero_day = statistics(pd.Series([100, 110, 110, 99], index=dates[:4]))
np.testing.assert_allclose(zero_day["odsetek dni wzrostowych"], 0.5, atol=1e-9)
tied_peak = statistics(pd.Series([100, 100, 90, 100], index=dates[:4]))
assert str(tied_peak["data szczytu UTC"]) == "2023-01-02" and str(tied_peak["data dołka UTC"]) == "2023-01-03"
assert str(tied_peak["powrót do szczytu UTC"]) == "2023-01-04"
no_dd = statistics(pd.Series([100, 110, 120], index=dates[:3]))
assert no_dd["maksymalne obsunięcie"] == 0.0 and no_dd["data szczytu UTC"] is None and no_dd["powrót do szczytu UTC"] is None
print("Wszystkie obliczenia kontrolne oraz test luki przeszły pomyślnie.")

Wszystkie obliczenia kontrolne oraz test luki przeszły pomyślnie.


## Dane rzeczywiste i wyniki

Statystyki obejmują tylko okres od 1.01.2023. W przypadku braków lub danych odrzuconych wyniki wymagają dodatkowej kontroli.

In [3]:
now = pd.Timestamp.now(tz="UTC")
data, checks = {}, {}
for symbol in ("BNBUSDT", "BTCUSDT"):
    data[symbol], checks[symbol] = download(symbol, now=now)
    c = checks[symbol]
    print(f"{symbol}: ostatni dostępny dzień UTC {data[symbol].index.max().date()}, oczekiwany {c['as_of'].date()}, brakujących dni {len(c['missing_dates'])}, niepoprawnych {c['invalid']}, zdublowanych {c['duplicate_rows']}, zmiany >50%: {len(c['outliers'])}")
    if c["invalid"] or c["duplicate_rows"] or c["unordered"] or len(c["missing_dates"]) or len(c["outliers"]):
        print("UWAGA: dane wymagają kontroli przed interpretacją.")
period = data["BNBUSDT"].loc["2023-01-01":]
result = statistics(period.close)
print(f"\nBNB/USDT: {period.index.min().date()} – {period.index.max().date()} UTC")
for label, value in result.items():
    formatted = "brak" if pd.isna(value) else f"{value:.10f}" if isinstance(value, (float, np.floating)) else str(value)
    print(f"{label}: {formatted}")

BNBUSDT: ostatni dostępny dzień UTC 2026-11-15, oczekiwany 2026-11-15, brakujących dni 0, niepoprawnych 0, zdublowanych 0, zmiany >50%: 0


BTCUSDT: ostatni dostępny dzień UTC 2026-11-15, oczekiwany 2026-11-15, brakujących dni 0, niepoprawnych 0, zdublowanych 0, zmiany >50%: 0

BNB/USDT: 2023-01-01 – 2026-11-15 UTC
liczba cen: 1415
liczba zwrotów dziennych: 1414
zwrot za okres: 1.4418576105
CAGR: 0.2591671832
maksymalne obsunięcie: -0.5819795013
data szczytu UTC: 2025-10-08
data dołka UTC: 2026-06-30
liczba dni od szczytu do dołka: 265
powrót do szczytu UTC: brak
średni zwrot prosty: 0.0009787540
mediana zwrotów prostych: 0.0011583658
najlepszy dzień: 0.1730232558
data najlepszego dnia UTC: 2024-03-13
najgorszy dzień: -0.1275976308
data najgorszego dnia UTC: 2026-02-05
odsetek dni wzrostowych: 0.5262411348
odchylenie logarytmicznych zwrotów (n-1): 0.0263372830
zmienność roczna: 0.5031730855
skośność: 0.0773765052
kurtoza: 4.3930199033


## Atak na Binance z 29.10.2026

[TRM Labs](https://www.trmlabs.com/resources/blog/north-korea-steals-usd-6-36-billion-from-binance-in-the-largest-hack-in-history) oszacował wartość środków, które wypłynęły z portfeli Binance, na około 6,36 mld USD. Poniżej porównujemy zamknięcia dziennych świec BNB i BTC. Świece pokazują zmianę od zamknięcia do zamknięcia. Analizę odchyleń BNB względem modelu z BTC zawiera notebook 05.

In [4]:
event_dates = pd.date_range("2026-10-28", "2026-10-30", freq="D", tz="UTC")
for symbol in ("BNBUSDT", "BTCUSDT"):
    close = data[symbol].close.reindex(event_dates)
    print(f"{symbol} — zamknięcia UTC (USDT):")
    for day, price in close.items():
        print(f"  {day.date()}: {price:.2f}" if pd.notna(price) else f"  {day.date()}: brak danych")
    for day in event_dates[1:]:
        previous = day - pd.Timedelta(days=1)
        if pd.notna(close.loc[day]) and pd.notna(close.loc[previous]):
            print(f"  zwrot {day.date()}: {(close.loc[day] / close.loc[previous] - 1) * 100:.2f}%")

BNBUSDT — zamknięcia UTC (USDT):
  2026-10-28: 759.85
  2026-10-29: 698.46
  2026-10-30: 704.42
  zwrot 2026-10-29: -8.08%
  zwrot 2026-10-30: 0.85%
BTCUSDT — zamknięcia UTC (USDT):
  2026-10-28: 81343.50
  2026-10-29: 77991.89
  2026-10-30: 77368.77
  zwrot 2026-10-29: -4.12%
  zwrot 2026-10-30: -0.80%
